# Cantor Basis 이해하기

이 노트북에서는 FAFFT(Frobenius Additive FFT)의 핵심 기반인 **Cantor Basis**를 단계적으로 이해합니다.

## 왜 Cantor Basis가 필요한가?

### 배경: 다항식 곱셈과 FFT

HQC, BIKE 같은 code-based 암호에서 가장 비용이 큰 연산은 **이진 다항식 곱셈** ($\mathbb{F}_2[x]$ 위의 곱셈)입니다.

일반적으로 큰 다항식을 빠르게 곱하려면 **FFT**를 사용합니다:
1. 다항식을 여러 점에서 **evaluate** (FFT)
2. 점별로 곱셈 (pointwise multiplication)
3. 결과를 다시 다항식으로 **interpolate** (inverse FFT)

### 문제: 일반 FFT는 $\mathbb{F}_2$에서 동작하지 않는다

일반 FFT (Cooley-Tukey)는 **primitive n-th root of unity** $\omega$ ($\omega^n = 1$, $\omega^k \neq 1$ for $0 < k < n$)가 필요합니다.

그런데 $\mathbb{F}_2 = \{0, 1\}$에서:
- $1^n = 1$이므로 $\omega = 1$밖에 없고, 이건 $\omega^1 = 1$이므로 trivial.
- 확장체 $\mathbb{F}_{2^m}$에서도 multiplicative group의 크기가 $2^m - 1$ (홀수)이므로, 2의 거듭제곱 크기의 FFT에 필요한 root of unity가 존재하지 않습니다.

### 해결: Additive FFT와 Cantor Basis

Additive FFT는 root of unity 대신 **$\mathbb{F}_{2^m}$의 additive 구조** (덧셈 부분군)를 이용합니다.

- 일반 FFT: multiplicative group의 coset 위에서 evaluate
- Additive FFT: **additive subspace의 coset** 위에서 evaluate

이때 additive subspace를 **재귀적으로 이분할**할 수 있어야 FFT의 butterfly 구조가 작동하는데,
**Cantor Basis**가 정확히 이런 재귀적 이분 구조를 제공합니다.

## 1. 준비: $\mathbb{F}_{2^m}$ 연산 구현

먼저 작은 binary extension field에서 실험하기 위한 도구를 만듭니다.

$\mathbb{F}_{2^m}$의 원소를 정수로 표현하고, 기약다항식(irreducible polynomial) 모듈로 연산합니다.

In [ ]:
class GF2m:
    """F_{2^m} 위의 연산을 지원하는 클래스.
    
    원소는 정수로 표현 (각 비트가 다항식의 계수).
    예: 0b1011 = x^3 + x + 1
    """
    def __init__(self, m, irr_poly):
        """
        m: 확장 차수
        irr_poly: 기약다항식 (정수 표현, 예: x^4+x+1 = 0b10011 = 19)
        """
        self.m = m
        self.irr = irr_poly
        self.size = 1 << m  # 2^m
    
    def add(self, a, b):
        """F_{2^m}에서의 덧셈 = XOR"""
        return a ^ b
    
    def mul(self, a, b):
        """F_{2^m}에서의 곱셈 (polynomial multiplication mod irr_poly)"""
        result = 0
        while b:
            if b & 1:
                result ^= a
            a <<= 1
            if a & self.size:  # degree >= m이면 reduction
                a ^= self.irr
            b >>= 1
        return result
    
    def sq(self, a):
        """a^2 (Frobenius map)"""
        return self.mul(a, a)
    
    def to_poly_str(self, a):
        """정수를 다항식 문자열로 변환"""
        if a == 0:
            return '0'
        terms = []
        for i in range(self.m):
            if a & (1 << i):
                if i == 0:
                    terms.append('1')
                elif i == 1:
                    terms.append('x')
                else:
                    terms.append(f'x^{i}')
        return ' + '.join(reversed(terms))

# F_{2^4} 생성: 기약다항식 x^4 + x + 1 (= 0b10011 = 19)
F16 = GF2m(4, 0b10011)

print(f"F_{{2^4}} = F_16: {F16.size}개 원소 (0 ~ {F16.size - 1})")
print(f"기약다항식: {F16.to_poly_str(0b10011)} = 0 (mod 연산에 사용)")
print()
print("연산 예시:")
print(f"  5 + 3 = {F16.to_poly_str(5)} + {F16.to_poly_str(3)} = {F16.to_poly_str(F16.add(5, 3))} = {F16.add(5, 3)}")
print(f"  5 * 3 = {F16.to_poly_str(5)} * {F16.to_poly_str(3)} = {F16.to_poly_str(F16.mul(5, 3))} = {F16.mul(5, 3)}")

## 2. Cantor Basis 정의

Cantor basis $\{v_0, v_1, \ldots, v_{m-1}\}$은 다음과 같이 **재귀적으로** 정의됩니다:

$$v_0 = 1$$
$$v_{i+1}^2 + v_{i+1} = v_i \quad \text{for } 0 \leq i < m-1$$

즉, $v_{i+1}$은 방정식 $x^2 + x = v_i$ ($\mathbb{F}_{2^m}$에서)의 한 해입니다.

> **주의**: $x^2 + x = v_i$가 $\mathbb{F}_{2^m}$에서 해를 갖기 위해서는 $m$이 충분히 커야 합니다.
> $x$가 해이면 $x+1$도 해입니다 ($\because (x+1)^2 + (x+1) = x^2 + 1 + x + 1 = x^2 + x$).

In [ ]:
def find_cantor_basis(field):
    """F_{2^m}에서 Cantor basis를 찾는다.
    
    v_0 = 1
    v_{i+1}^2 + v_{i+1} = v_i 를 만족하는 v_{i+1}을 brute-force로 탐색.
    """
    m = field.m
    basis = [0] * m
    basis[0] = 1  # v_0 = 1
    
    for i in range(m - 1):
        target = basis[i]  # v_i
        found = False
        for x in range(field.size):
            # x^2 + x = target 인 x를 찾는다
            if field.add(field.sq(x), x) == target:
                basis[i + 1] = x
                found = True
                break
        if not found:
            print(f"경고: v_{i+1}을 찾을 수 없음 (x^2 + x = {target}의 해가 F_{{2^{m}}}에 없음)")
            return None
    
    return basis

# F_{2^4}에서 Cantor basis 찾기
cantor = find_cantor_basis(F16)

if cantor:
    print("Cantor Basis for F_{2^4}:")
    print("=" * 50)
    for i, v in enumerate(cantor):
        print(f"  v_{i} = {v:4d} = 0b{v:04b} = {F16.to_poly_str(v)}")
    
    print("\n검증 (v_{i+1}^2 + v_{i+1} = v_i):")
    print("=" * 50)
    for i in range(len(cantor) - 1):
        v_next = cantor[i + 1]
        lhs = F16.add(F16.sq(v_next), v_next)
        print(f"  v_{i+1}^2 + v_{i+1} = {lhs} == v_{i} = {cantor[i]}  ✓" if lhs == cantor[i] else f"  FAIL!")

## 3. Cantor Basis가 만드는 부분공간 $V_i$

Cantor basis의 핵심은 **부분공간(subspace)의 재귀적 구조**입니다.

$$V_i = \text{span}\{v_0, v_1, \ldots, v_{i-1}\} = \left\{\sum_{j=0}^{i-1} c_j v_j \;\middle|\; c_j \in \{0,1\}\right\}$$

- $V_0 = \{0\}$
- $V_1 = \{0, v_0\} = \{0, 1\}$
- $V_2 = \{0, v_0, v_1, v_0 + v_1\} = \{0, 1, v_1, v_1 + 1\}$
- ...
- $V_m = \mathbb{F}_{2^m}$ 전체

**중요**: $|V_i| = 2^i$이고, $V_i$는 $\mathbb{F}_2$-선형 부분공간 (덧셈에 닫혀 있음).

### 이분 구조

$$V_i = V_{i-1} \cup (v_{i-1} + V_{i-1})$$

이것이 FFT의 butterfly에서 문제를 반으로 나누는 데 사용됩니다.

In [ ]:
def compute_subspaces(field, basis):
    """Cantor basis로부터 부분공간 V_0, V_1, ..., V_m을 계산."""
    m = field.m
    subspaces = []
    
    # V_0 = {0}
    V = {0}
    subspaces.append(frozenset(V))
    
    for i in range(m):
        # V_{i+1} = V_i ∪ (v_i + V_i)
        V_new = set(V)
        for elem in V:
            V_new.add(field.add(basis[i], elem))
        V = V_new
        subspaces.append(frozenset(V))
    
    return subspaces

subspaces = compute_subspaces(F16, cantor)

print("부분공간 V_i:")
print("=" * 60)
for i, V in enumerate(subspaces):
    elems = sorted(V)
    print(f"  V_{i} (|V_{i}| = {len(V):2d}): {elems}")

print("\n이분 구조 확인: V_i = V_{i-1} ∪ (v_{i-1} + V_{i-1})")
print("=" * 60)
for i in range(1, len(subspaces)):
    V_prev = subspaces[i - 1]
    coset = frozenset(F16.add(cantor[i - 1], elem) for elem in V_prev)
    union = V_prev | coset
    match = union == subspaces[i]
    print(f"  V_{i} = V_{i-1} ∪ (v_{i-1} + V_{i-1})  →  {sorted(V_prev)} ∪ {sorted(coset)}  ✓" if match else f"  FAIL!")

## 4. Vanishing Polynomial $s_i(x)$

**Vanishing polynomial** $s_i(x)$는 부분공간 $V_i$의 모든 원소를 0으로 보내는 다항식입니다:

$$s_i(x) = \prod_{j \in V_i} (x - j) = \prod_{j \in V_i} (x + j) \quad (\text{since } -1 = 1 \text{ in } \mathbb{F}_2)$$

재귀적 정의:
$$s_0(x) = x$$
$$s_1(x) = x^2 + x$$
$$s_i(x) = s_{i-1}(s_1(x)) \quad \text{for } i > 1$$

### 핵심 성질

1. **선형성**: $s_i(a + b) = s_i(a) + s_i(b)$ — 이것이 additive FFT를 가능하게 하는 핵심!
2. **$\deg s_i = 2^i$**
3. **$V_i$의 모든 원소에서 0**: $s_i(v) = 0$ for all $v \in V_i$
4. **Evaluation on basis**: $s_i(v_i) = v_{i-1}$ (논문의 relation $s_1(v_i) = v_i^2 - v_i = v_{i-1}$에서 유도)

In [ ]:
def vanishing_poly(field, x, i):
    """s_i(x)를 재귀적으로 계산.
    
    s_0(x) = x
    s_1(x) = x^2 + x
    s_i(x) = s_{i-1}(s_1(x)) for i > 1
    """
    if i == 0:
        return x
    # s_1(x) = x^2 + x
    s1_x = field.add(field.sq(x), x)
    if i == 1:
        return s1_x
    return vanishing_poly(field, s1_x, i - 1)

print("Vanishing polynomial s_i(x) 검증")
print("=" * 60)

for i in range(1, F16.m + 1):
    # s_i가 V_i의 모든 원소를 0으로 보내는지 확인
    all_zero = all(vanishing_poly(F16, v, i) == 0 for v in subspaces[i])
    # V_i 밖의 원소는 0이 아닌지 확인
    non_zero_outside = all(
        vanishing_poly(F16, v, i) != 0 
        for v in range(F16.size) if v not in subspaces[i]
    )
    print(f"  s_{i}(x): V_{i}의 모든 원소 → 0: {'✓' if all_zero else '✗'},  V_{i} 밖 → ≠0: {'✓' if non_zero_outside else '✗'}")

print("\n선형성 검증: s_i(a + b) = s_i(a) + s_i(b)")
print("=" * 60)
import random
random.seed(42)
for i in range(1, F16.m + 1):
    tests_passed = 0
    total_tests = 50
    for _ in range(total_tests):
        a = random.randint(0, F16.size - 1)
        b = random.randint(0, F16.size - 1)
        lhs = vanishing_poly(F16, F16.add(a, b), i)
        rhs = F16.add(vanishing_poly(F16, a, i), vanishing_poly(F16, b, i))
        if lhs == rhs:
            tests_passed += 1
    print(f"  s_{i}: {tests_passed}/{total_tests} 테스트 통과 {'✓' if tests_passed == total_tests else '✗'}")

print("\ns_1(v_i) = v_{i-1} 검증 (Cantor basis의 정의와 일치):")
print("=" * 60)
for i in range(1, F16.m):
    result = vanishing_poly(F16, cantor[i], 1)  # s_1(v_i) = v_i^2 + v_i
    print(f"  s_1(v_{i}) = v_{i}^2 + v_{i} = {result} == v_{i-1} = {cantor[i-1]}  {'✓' if result == cantor[i-1] else '✗'}")

## 5. 왜 이 구조가 FFT를 가능하게 하는가?

### 일반 FFT의 핵심 원리 (복습)

길이 $n$인 다항식 $f(x)$를 $n$개의 점에서 evaluate:
- 점들: $\{1, \omega, \omega^2, \ldots, \omega^{n-1}\}$ (root of unity)
- $\omega^{n/2} = -1$이므로, 점들이 $\{\omega^{\text{even}}\}$과 $\{\omega^{\text{odd}}\}$로 이분됨
- $f(x) = f_{\text{even}}(x^2) + x \cdot f_{\text{odd}}(x^2)$로 분할 → 재귀

### Additive FFT의 원리

길이 $2^i$인 다항식을 **affine subspace** $a + V_i$ 위에서 evaluate:

$$\text{addFFT}(f, a + V_i) \mapsto (f(a), f(a+1), f(a+v_1), \ldots)$$

**이분 구조**: $V_i = V_{i-1} \cup (v_{i-1} + V_{i-1})$이므로:

$$a + V_i = (a + V_{i-1}) \cup (a + v_{i-1} + V_{i-1})$$

**Butterfly 연산**: 다항식 $f = f_l + s_{i-1} \cdot f_h$로 분할하면:
- $a + V_{i-1}$ 위에서: $f_l + s_{i-1}(a) \cdot f_h$를 evaluate
- $a + v_{i-1} + V_{i-1}$ 위에서: $f_l + (s_{i-1}(a) + 1) \cdot f_h$를 evaluate

$s_{i-1}$의 **선형성** 덕분에 $s_{i-1}(a+v) = s_{i-1}(a) + s_{i-1}(v)$이고,
$v \in V_{i-1}$이면 $s_{i-1}(v) = 0$이므로 상수 $s_{i-1}(a)$만 곱하면 됩니다.

이것이 butterfly에서 **1회의 곱셈**만 필요한 이유입니다.

In [ ]:
print("Butterfly 구조 시각화")
print("=" * 60)
print()
print("일반 FFT (Cooley-Tukey):")
print("  evaluation points: {1, ω, ω², ..., ω^(n-1)}")
print("  분할 기준: ω^(n/2) = -1 (multiplicative 구조)")
print("  butterfly: f(ω^k) = f_even(ω^2k) + ω^k · f_odd(ω^2k)")
print()
print("Additive FFT:")
print("  evaluation points: a + V_i (additive coset)")
print("  분할 기준: V_i = V_{i-1} ∪ (v_{i-1} + V_{i-1}) (additive 구조)")
print("  butterfly: f = f_l + s_{i-1} · f_h")
print("    → f_l + s_{i-1}(a)·f_h       on a + V_{i-1}")
print("    → f_l + (s_{i-1}(a)+1)·f_h   on a + v_{i-1} + V_{i-1}")
print()
print("=" * 60)
print("비교:")
print(f"{'':>20} {'일반 FFT':>20} {'Additive FFT':>20}")
print(f"{'체(Field)':>20} {'C, F_p (p홀수)':>20} {'F_{2^m}':>20}")
print(f"{'evaluation 점':>20} {'root of unity':>20} {'additive coset':>20}")
print(f"{'분할 구조':>20} {'multiplicative':>20} {'additive':>20}")
print(f"{'butterfly 곱셈':>20} {'twiddle factor':>20} {'s_{i-1}(a)':>20}")
print(f"{'필요 조건':>20} {'ω^n = 1':>20} {'Cantor basis':>20}")

## 6. 간단한 Additive FFT 구현

위의 개념을 $\mathbb{F}_{2^4}$에서 실제로 구현해봅니다.

addFFT는 두 단계로 구성됩니다:
1. **BasisCvt**: monomial basis → polynomial basis $\{1, X_1, X_2, \ldots\}$ 변환
2. **Butterfly**: 재귀적 butterfly 연산으로 evaluation 수행

여기서는 이해를 위해 butterfly만 구현합니다 (이미 polynomial basis라고 가정).

In [ ]:
def additive_fft_butterfly(field, basis, coeffs, a, depth):
    """Additive FFT의 butterfly 연산.
    
    Parameters:
        field: GF2m 인스턴스
        basis: Cantor basis
        coeffs: polynomial basis에서의 계수 리스트 (길이 2^depth)
        a: evaluation domain의 시작점 (a + V_depth에서 evaluate)
        depth: 현재 재귀 깊이 (= i, V_i 위에서 evaluate)
    
    Returns:
        f(a+0), f(a+v_0), f(a+v_1), ... 의 리스트
    """
    n = len(coeffs)
    
    # Base case: 상수 다항식
    if n == 1:
        return [coeffs[0]]
    
    half = n // 2
    
    # f = f_l + X_{n/2} · f_h 에서 f_l, f_h 분리
    f_l = list(coeffs[:half])
    f_h = list(coeffs[half:])
    
    # s_{depth-1}(a) 계산 — butterfly에서의 곱셈 상수
    s_val = vanishing_poly(field, a, depth - 1)
    
    # f_l' = f_l + s_{depth-1}(a) · f_h       (왼쪽 절반)
    # f_h' = f_l + (s_{depth-1}(a) + 1) · f_h  (오른쪽 절반)
    left_coeffs = [field.add(f_l[j], field.mul(s_val, f_h[j])) for j in range(half)]
    right_coeffs = [field.add(f_l[j], field.mul(field.add(s_val, 1), f_h[j])) for j in range(half)]
    
    # 재귀: 왼쪽은 a + V_{depth-1}, 오른쪽은 (a + v_{depth-1}) + V_{depth-1}
    left_result = additive_fft_butterfly(field, basis, left_coeffs, a, depth - 1)
    right_result = additive_fft_butterfly(field, basis, right_coeffs, field.add(a, basis[depth - 1]), depth - 1)
    
    return left_result + right_result


# 테스트: 간단한 다항식을 evaluate하고 직접 계산과 비교
# polynomial basis에서의 계수: f = c_0 + c_1·X_1 + c_2·X_2 + c_3·X_3
# 여기서는 간단히 f = 3 + 5·X_1 + 7·X_2 + 2·X_3 (polynomial basis 기준)
coeffs = [3, 5, 7, 2]  # 4 = 2^2 개 계수
a = 0  # V_2 위에서 evaluate
depth = 2  # V_2 = {0, 1, v_1, v_1+1}

# Polynomial basis의 X_k(x) 계산
def poly_basis_X(field, x, k):
    """X_k(x) = product of s_j(x) for j where k has bit j set."""
    result = 1
    j = 0
    while k > 0:
        if k & 1:
            result = field.mul(result, vanishing_poly(field, x, j))
        k >>= 1
        j += 1
    return result

def eval_poly_basis(field, coeffs, x):
    """polynomial basis에서 f(x) = sum(c_k * X_k(x)) 계산."""
    result = 0
    for k, c in enumerate(coeffs):
        result = field.add(result, field.mul(c, poly_basis_X(field, x, k)))
    return result

# V_2의 원소들
eval_points = sorted(subspaces[depth])
print(f"Evaluation domain: a + V_{depth} = {eval_points}")
print(f"Polynomial basis 계수: {coeffs}")
print()

# additive FFT로 계산
fft_result = additive_fft_butterfly(F16, cantor, coeffs, a, depth)

# 직접 evaluate로 검증
print(f"{'점':>6} {'FFT 결과':>10} {'직접 계산':>10} {'일치':>6}")
print("-" * 40)
for idx, pt in enumerate(eval_points):
    direct = eval_poly_basis(F16, coeffs, pt)
    match = fft_result[idx] == direct
    print(f"{pt:>6} {fft_result[idx]:>10} {direct:>10} {'✓' if match else '✗':>6}")

## 7. 정리: Cantor Basis → Additive FFT → FAFFT

### 지금까지 배운 것

```
Cantor Basis {v_0, v_1, ..., v_{m-1}}
    │
    ├── 부분공간 V_i = span{v_0,...,v_{i-1}}
    │       │
    │       └── 이분 구조: V_i = V_{i-1} ∪ (v_{i-1} + V_{i-1})
    │                              → FFT butterfly 구조
    │
    └── Vanishing polynomial s_i(x)
            │
            ├── 선형성: s_i(a+b) = s_i(a) + s_i(b)
            │          → butterfly에서 상수 곱셈만 필요
            │
            └── 재귀성: s_i = s_{i-1} ∘ s_1
                       → 분할 정복 가능
```

### 다음 단계: FAFFT

Additive FFT는 $2^l$개 점에서 evaluate합니다. 다항식 곱셈에는 $2n$개 점이 필요한데,
FAFFT는 **Frobenius map** ($x \mapsto x^2$)을 이용해 evaluation point 수를 $m$배 줄입니다:

- Additive FFT: $2n$개 점에서 evaluate 필요
- FAFFT: $2n/m$개 점에서만 evaluate → 나머지는 Frobenius map으로 유도

이것이 FAFFT가 더 빠른 이유이며, 다음 노트북에서 다룹니다.